In [11]:
import joblib
import numpy as np
from tensorflow.keras.models import load_model

In [4]:
# XGBoost 
xgb = joblib.load("xgb_multi_risk_model.pkl")

# LSTM
lstm = load_model("lstm_model.h5", compile=False)
lstm.compile(optimizer='adam', loss='mse')

# CNN
cnn = load_model("cnn_1d_model.h5", compile=False)
cnn.compile(optimizer='adam', loss='mse')

In [14]:
scaler_X = joblib.load("scaler_X.pkl")
scaler_y = joblib.load("scaler_y.pkl")

cnn_scaler_X = joblib.load("cnn_scaler_X.pkl")
cnn_scaler_y = joblib.load("cnn_scaler_y.pkl")

In [15]:
def hybrid_predict(current_features, past_7_days):

    # XGBoost
    xgb_pred = xgb.predict(current_features.reshape(1, -1))

    # LSTM
    seq_scaled = scaler_X.transform(past_7_days)
    seq_scaled = seq_scaled.reshape(1, 7, -1)

    lstm_pred_scaled = lstm.predict(seq_scaled)
    lstm_pred = scaler_y.inverse_transform(lstm_pred_scaled)

    # CNN
    cnn_seq_scaled = cnn_scaler_X.transform(past_7_days)
    cnn_seq_scaled = cnn_seq_scaled.reshape(1, 7, -1)

    cnn_pred_scaled = cnn.predict(cnn_seq_scaled)
    cnn_pred = cnn_scaler_y.inverse_transform(cnn_pred_scaled)

    # Combine
    final_pred = (
        0.4 * xgb_pred +
        0.3 * lstm_pred +
        0.3 * cnn_pred
    )

    return final_pred

In [16]:
import pandas as pd

In [17]:
df = pd.read_csv("data.csv")

In [18]:
features = df.drop(columns=[
    "date",
    "flood_risk",
    "landslide_risk",
    "wildfire_risk",
    "glacier_melt_risk"
])

In [22]:
i = 50

past_7_days = features.iloc[i-7:i].values
current_features = features.iloc[i].values

pred = hybrid_predict(current_features, past_7_days)

realistic_pred = np.clip(realistic_pred, 0, 1)
print(f"Flood Risk: {realistic_pred[0]:.2f}")
print(f"Landslide Risk: {realistic_pred[1]:.2f}")
print(f"Wildfire Risk: {realistic_pred[2]:.2f}")
print(f"Glacier Melt Risk: {realistic_pred[3]:.2f}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
Flood Risk: 0.88
Landslide Risk: 0.83
Wildfire Risk: 0.86
Glacier Melt Risk: 0.83


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


In [23]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Load dataset
df = pd.read_csv("data.csv")

# Prepare features
features = df.drop(columns=[
    "date",
    "flood_risk",
    "landslide_risk",
    "wildfire_risk",
    "glacier_melt_risk"
])

targets = df[[
    "flood_risk",
    "landslide_risk",
    "wildfire_risk",
    "glacier_melt_risk"
]].values

# Store predictions
predictions = []
actuals = []

# Loop through dataset (skip first 7 for sequence)
for i in range(7, len(features)):
    
    past_7_days = features.iloc[i-7:i].values
    current_features = features.iloc[i].values
    
    pred = hybrid_predict(current_features, past_7_days)[0]
    
    predictions.append(pred)
    actuals.append(targets[i])

predictions = np.array(predictions)
actuals = np.array(actuals)

# -----------------------------
# 📊 Metrics
# -----------------------------
rmse = np.sqrt(mean_squared_error(actuals, predictions))
mae = mean_absolute_error(actuals, predictions)

print("\n📊 HYBRID MODEL PERFORMANCE:")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feat

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step

...

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feat

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feat

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step

📊 HYBRID MODEL PERFORMANCE:
RMSE: 0.0311
MAE: 0.0116


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


In [26]:
# Convert to binary (threshold = 0.6)
pred_binary = (predictions > 0.6).astype(int)
actual_binary = (actuals > 0.6).astype(int)

from sklearn.metrics import accuracy_score

accuracy = accuracy_score(actual_binary.flatten(), pred_binary.flatten())

print(f"\nClassification Accuracy: {accuracy:.2f}")



 Classification Accuracy: 93.0976479
